# Greedy Horner Factorization in Egglog

Code for ... blog post

In [1]:
def bending_function(Q, Bp, Bpp):
    xp = Q.__array_namespace__()
    QM = xp.reshape(Q, (4, 3)).T

    yip = xp.vecdot(QM, Bp)
    yipp = xp.vecdot(QM, Bpp)
    num = xp.linalg.vector_norm(xp.cross(yip, yipp))
    den = xp.linalg.vector_norm(yip) ** 3
    return (num / den) ** 2


if __name__ == "__main__":
    import numpy as np

    # create random array inputs
    ng = np.random.default_rng()
    # control points
    q_np = ng.random(12).astype(float)
    # bernstein coefficients
    bp_np = ng.random(4).astype(float)
    bpp_np = ng.random(4).astype(float)
    # Try on random inputs
    bending_function(q_np, bp_np, bpp_np)

In [2]:
import egglog
import egglog.exp.array_api as enp

Bp = enp.NDArray([enp.Value.var(f"bp{i}") for i in range(1, 5)])
Bpp = enp.NDArray([enp.Value.var(f"bpp{i}") for i in range(1, 5)])
Q = enp.NDArray([enp.Value.var(f"q{i}") for i in range(1, 13)])
FunctionBending = enp.NDArray(bending_function(Q, Bp, Bpp).eval())
GradientBending = enp.NDArray(FunctionBending.diff(Q).eval())

In [3]:
FunctionBending

_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
NDArray(
    RecursiveValue(
        (
            (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
            + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
            + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
        )
        / (
            _Value_6 ** Value.from_int(Int(2))
            + _Value_1 ** Value.from_int(Int(2))
            + _Value_3 ** Value.from_int(Int(2))
        )
        ** Value.from_int(Int(3))
    )
)

In [4]:
import time
from dataclasses import dataclass
from collections.abc import Callable


@dataclass
class Report:
    register_sec: int
    run_sec: int
    extract_sec: int
    extracted: enp.NDArray
    cost: int
    function_sizes: list[tuple[Callable, int]]
    updated: bool

    @property
    def total_sec(self) -> int:
        return self.register_sec + self.run_sec + self.extract_sec

    @property
    def total_size(self) -> int:
        return sum(size for _, size in self.function_sizes)


def run_example(ruleset: egglog.Ruleset, input: enp.NDArray, egraph: egglog.EGraph | None = None) -> Report:
    if egraph is None:
        egraph = egglog.EGraph()
    start = time.perf_counter()

    egraph.register(input)
    register_sec = time.perf_counter() - start
    start = time.perf_counter()
    report = egraph.run(ruleset)
    run_sec = time.perf_counter() - start
    start = time.perf_counter()
    extracted, cost = egraph.extract(input, include_cost=True)
    extract_sec = time.perf_counter() - start
    return Report(register_sec, run_sec, extract_sec, extracted, cost, egraph.all_function_sizes(), report.updated)

In [5]:
@egglog.ruleset
def remove_subtraction(a: enp.Value, b: enp.Value):
    yield egglog.rewrite(a - b, subsume=True).to(a + (-1) * b)


@egglog.ruleset
def distribute(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.rewrite((a + b) * c, subsume=True).to(a * c + b * c)
    yield egglog.rewrite(c * (a + b), subsume=True).to(c * a + c * b)


@egglog.ruleset
def factoring(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.birewrite((a + b) * c).to(a * c + b * c)
    yield egglog.rewrite(a * b).to(b * a)
    yield egglog.rewrite(a + b).to(b + a)
    yield egglog.birewrite(a * (b * c)).to((a * b) * c)


@dataclass
class TotalReport:
    original: Report
    distributed: Report
    factored: list[Report]
    polynomial_multisets: Report
    polynomial_multisets_factored: Report
    polynomial: Report

    @property
    def combined_factored(self) -> Report:
        return Report(
            register_sec=self.factored[0].register_sec,
            run_sec=sum(r.run_sec for r in self.factored),
            extract_sec=self.factored[-1].extract_sec,
            extracted=self.factored[-1].extracted,
            cost=self.factored[-1].cost,
            # combine all
            function_sizes=self.factored[-1].function_sizes,
            updated=self.factored[-1].updated,
        )

    @property
    def combined_polynomial(self) -> Report:
        return Report(
            register_sec=self.polynomial_multisets.register_sec,
            run_sec=self.polynomial_multisets.run_sec
            + self.polynomial_multisets_factored.run_sec
            + self.polynomial.run_sec,
            extract_sec=self.polynomial.run_sec,
            extracted=self.polynomial.extracted,
            cost=self.polynomial.cost,
            function_sizes=self.polynomial.function_sizes,
            updated=self.polynomial.updated,
        )

    def __str__(self) -> str:
        return f"""Costs:
* original: {self.original.cost:,}
* distributed: {self.distributed.cost:,}
* factored: {self.combined_factored.cost:,}
* horner multisets: {self.combined_polynomial.cost:,}


Number of nodes:
* original: {self.original.total_size:,}
* distributed: {self.distributed.total_size:,}
* factored: {self.combined_factored.total_size:,}
* horner multisets: {self.combined_polynomial.total_size:,}

Time:
* original: {self.original.total_sec:.2f}s
* distributed: {self.distributed.total_sec:.2f}s
* factored: {self.combined_factored.total_sec:.2f}s
* horner multisets: {self.combined_polynomial.total_sec:.2f}s
"""


def try_example(expr: enp.NDArray):
    original_report = run_example(remove_subtraction, expr)
    print(f"original cost: {original_report.cost:,}")
    distributed_report = run_example(distribute.saturate(), original_report.extracted)
    print(f"distributed cost: {distributed_report.cost:,}")

    egraph = egglog.EGraph()
    polynomial_multisets_report = run_example(
        enp.to_polynomial_ruleset.saturate(), distributed_report.extracted, egraph
    )
    polynomial_multisets_factored_report = run_example(
        enp.factor_ruleset.saturate(), polynomial_multisets_report.extracted, egraph
    )
    polynomial_report = run_example(
        enp.from_polynomial_ruleset.saturate(), polynomial_multisets_factored_report.extracted, egraph
    )
    print(f"polynomial cost: {polynomial_report.cost:,}")

    egraph = egglog.EGraph()
    factored_reports = []
    for i in range(20):
        res = run_example(factoring, distributed_report.extracted, egraph)
        if not res.updated or res.run_sec > 10:
            break
        print(f"factoring iteration {i}, cost: {res.cost:,}")
        factored_reports.append(res)
    print("Finished\n\n")
    return TotalReport(
        original_report,
        distributed_report,
        factored_reports,
        polynomial_multisets_report,
        polynomial_multisets_factored_report,
        polynomial_report,
    )

In [6]:
(function_example := try_example(FunctionBending))
print(function_example)

original cost: 401
distributed cost: 1,445
polynomial cost: 401
factoring iteration 0, cost: 1,445
factoring iteration 1, cost: 1,325
factoring iteration 2, cost: 1,133
factoring iteration 3, cost: 941
factoring iteration 4, cost: 857
factoring iteration 5, cost: 773
factoring iteration 6, cost: 617
factoring iteration 7, cost: 473
factoring iteration 8, cost: 401
factoring iteration 9, cost: 401
factoring iteration 10, cost: 401
factoring iteration 11, cost: 401
factoring iteration 12, cost: 401
factoring iteration 13, cost: 401
factoring iteration 14, cost: 401
factoring iteration 15, cost: 401
Finished


Costs:
* original: 401
* distributed: 1,445
* factored: 401
* horner multisets: 401


Number of nodes:
* original: 97
* distributed: 904
* factored: 13,040
* horner multisets: 916

Time:
* original: 0.01s
* distributed: 0.01s
* factored: 0.09s
* horner multisets: 0.04s



In [7]:
(gradient_example := try_example(GradientBending))
print(gradient_example)

original cost: 20,570
distributed cost: 4,250,786
polynomial cost: 80,025
factoring iteration 0, cost: 3,976,490
factoring iteration 1, cost: 3,431,213
factoring iteration 2, cost: 2,126,268
Finished


Costs:
* original: 20,570
* distributed: 4,250,786
* factored: 2,126,268
* horner multisets: 80,025


Number of nodes:
* original: 470
* distributed: 588,125
* factored: 2,583,064
* horner multisets: 109,746

Time:
* original: 0.03s
* distributed: 3.56s
* factored: 12.94s
* horner multisets: 7.70s



In [8]:
function_example.distributed.extracted

_Value_1 = Value.var("q2") * Value.var("bp1")
_Value_2 = Value.var("q3") * Value.var("bpp1")
_Value_3 = Value.var("q6") * Value.var("bpp2")
_Value_4 = Value.var("q9") * Value.var("bpp3")
_Value_5 = Value.var("q12") * Value.var("bpp4")
_Value_6 = Value.var("q5") * Value.var("bp2")
_Value_7 = Value.var("q8") * Value.var("bp3")
_Value_8 = Value.var("q11") * Value.var("bp4")
_Value_9 = Value.from_int(Int(-1))
_Value_10 = Value.var("q3") * Value.var("bp1")
_Value_11 = Value.var("q2") * Value.var("bpp1")
_Value_12 = Value.var("q5") * Value.var("bpp2")
_Value_13 = Value.var("q8") * Value.var("bpp3")
_Value_14 = Value.var("q11") * Value.var("bpp4")
_Value_15 = Value.var("q6") * Value.var("bp2")
_Value_16 = Value.var("q9") * Value.var("bp3")
_Value_17 = Value.var("q12") * Value.var("bp4")
_Value_18 = Value.var("q1") * Value.var("bpp1")
_Value_19 = Value.var("q4") * Value.var("bpp2")
_Value_20 = Value.var("q7") * Value.var("bpp3")
_Value_21 = Value.var("q10") * Value.var("bpp4")
_Value_22 = Value.var("q1") * Value.var("bp1")
_Value_23 = Value.var("q4") * Value.var("bp2")
_Value_24 = Value.var("q7") * Value.var("bp3")
_Value_25 = Value.var("q10") * Value.var("bp4")
NDArray(
    RecursiveValue(
        (
            (
                _Value_1 * _Value_2
                + _Value_1 * _Value_3
                + _Value_1 * _Value_4
                + _Value_1 * _Value_5
                + (
                    _Value_6 * _Value_2
                    + _Value_6 * _Value_3
                    + _Value_6 * _Value_4
                    + _Value_6 * _Value_5
                )
                + (
                    _Value_7 * _Value_2
                    + _Value_7 * _Value_3
                    + _Value_7 * _Value_4
                    + _Value_7 * _Value_5
                )
                + (
                    _Value_8 * _Value_2
                    + _Value_8 * _Value_3
                    + _Value_8 * _Value_4
                    + _Value_8 * _Value_5
                )
                + (
                    _Value_9 * (_Value_10 * _Value_11)
                    + _Value_9 * (_Value_10 * _Value_12)
                    + _Value_9 * (_Value_10 * _Value_13)
                    + _Value_9 * (_Value_10 * _Value_14)
                    + (
                        _Value_9 * (_Value_15 * _Value_11)
                        + _Value_9 * (_Value_15 * _Value_12)
                        + _Value_9 * (_Value_15 * _Value_13)
                        + _Value_9 * (_Value_15 * _Value_14)
                    )
                    + (
                        _Value_9 * (_Value_16 * _Value_11)
                        + _Value_9 * (_Value_16 * _Value_12)
                        + _Value_9 * (_Value_16 * _Value_13)
                        + _Value_9 * (_Value_16 * _Value_14)
                    )
                    + (
                        _Value_9 * (_Value_17 * _Value_11)
                        + _Value_9 * (_Value_17 * _Value_12)
                        + _Value_9 * (_Value_17 * _Value_13)
                        + _Value_9 * (_Value_17 * _Value_14)
                    )
                )
            )
            ** Value.from_int(Int(2))
            + (
                _Value_10 * _Value_18
                + _Value_10 * _Value_19
                + _Value_10 * _Value_20
                + _Value_10 * _Value_21
                + (
                    _Value_15 * _Value_18
                    + _Value_15 * _Value_19
                    + _Value_15 * _Value_20
                    + _Value_15 * _Value_21
                )
                + (
                    _Value_16 * _Value_18
                    + _Value_16 * _Value_19
                    + _Value_16 * _Value_20
                    + _Value_16 * _Value_21
                )
                + (
                    _Value_17 * _Value_18
                    + _Value_17 * _Value_19
                    + _Value_17 * _Value_20
                    + _Value_17 * _Value_21
                )
                + (
  

In [15]:
function_example.polynomial_multisets.extracted

_Value_1 = Value.var("bpp1")
_Value_2 = Value.var("bp1")
_Value_3 = Value.var("bpp2")
_Value_4 = Value.var("bpp3")
_Value_5 = Value.var("q12")
_Value_6 = Value.var("bpp4")
_Value_7 = Value.var("bp2")
_Value_8 = Value.var("bp3")
_Value_9 = Value.var("bp4")
_Value_10 = Value.var("q11")
_Value_11 = Value.from_int(Int(-1))
_Value_12 = polynomial(
    MultiSet(
        MultiSet(Value.var("q3"), _Value_1, Value.var("q2"), _Value_2),
        MultiSet(Value.var("q6"), _Value_3, Value.var("q2"), _Value_2),
        MultiSet(Value.var("q2"), Value.var("q9"), _Value_4, _Value_2),
        MultiSet(Value.var("q2"), _Value_2, _Value_5, _Value_6),
        MultiSet(_Value_7, Value.var("q3"), _Value_1, Value.var("q5")),
        MultiSet(_Value_7, Value.var("q6"), _Value_3, Value.var("q5")),
        MultiSet(_Value_7, Value.var("q9"), _Value_4, Value.var("q5")),
        MultiSet(_Value_7, Value.var("q5"), _Value_5, _Value_6),
        MultiSet(Value.var("q3"), _Value_1, _Value_8, Value.var("q8")),
        MultiSet(Value.var("q6"), _Value_3, _Value_8, Value.var("q8")),
        MultiSet(_Value_8, Value.var("q9"), _Value_4, Value.var("q8")),
        MultiSet(_Value_8, _Value_5, _Value_6, Value.var("q8")),
        MultiSet(Value.var("q3"), _Value_1, _Value_9, _Value_10),
        MultiSet(Value.var("q6"), _Value_3, _Value_9, _Value_10),
        MultiSet(Value.var("q9"), _Value_4, _Value_9, _Value_10),
        MultiSet(_Value_9, _Value_5, _Value_6, _Value_10),
        MultiSet(Value.var("q3"), _Value_1, _Value_11, Value.var("q2"), _Value_2),
        MultiSet(Value.var("q3"), _Value_11, _Value_3, Value.var("q5"), _Value_2),
        MultiSet(Value.var("q3"), _Value_11, _Value_4, _Value_2, Value.var("q8")),
        MultiSet(Value.var("q3"), _Value_11, _Value_2, _Value_6, _Value_10),
        MultiSet(_Value_7, _Value_1, _Value_11, Value.var("q6"), Value.var("q2")),
        MultiSet(_Value_7, _Value_11, Value.var("q6"), _Value_3, Value.var("q5")),
        MultiSet(_Value_7, _Value_11, Value.var("q6"), _Value_4, Value.var("q8")),
        MultiSet(_Value_7, _Value_11, Value.var("q6"), _Value_6, _Value_10),
        MultiSet(_Value_1, _Value_11, _Value_8, Value.var("q2"), Value.var("q9")),
        MultiSet(_Value_11, _Value_3, _Value_8, Value.var("q9"), Value.var("q5")),
        MultiSet(_Value_11, _Value_8, Value.var("q9"), _Value_4, Value.var("q8")),
        MultiSet(_Value_11, _Value_8, Value.var("q9"), _Value_6, _Value_10),
        MultiSet(_Value_1, _Value_11, Value.var("q2"), _Value_9, _Value_5),
        MultiSet(_Value_11, _Value_3, Value.var("q5"), _Value_9, _Value_5),
        MultiSet(_Value_11, _Value_4, _Value_9, _Value_5, Value.var("q8")),
        MultiSet(_Value_11, _Value_9, _Value_5, _Value_6, _Value_10),
    )
)
_Value_13 = Value.var("q10")
_Value_14 = polynomial(
    MultiSet(
        MultiSet(Value.var("q3"), _Value_1, _Value_2, Value.var("q1")),
        MultiSet(Value.var("q4"), Value.var("q3"), _Value_3, _Value_2),
        MultiSet(Value.var("q3"), Value.var("q7"), _Value_4, _Value_2),
        MultiSet(Value.var("q3"), _Value_13, _Value_2, _Value_6),
        MultiSet(_Value_7, _Value_1, Value.var("q6"), Value.var("q1")),
        MultiSet(Value.var("q4"), _Value_7, Value.var("q6"), _Value_3),
        MultiSet(_Value_7, Value.var("q6"), Value.var("q7"), _Value_4),
        MultiSet(_Value_7, Value.var("q6"), _Value_13, _Value_6),
        MultiSet(_Value_1, _Value_8, Value.var("q9"), Value.var("q1")),
        MultiSet(Value.var("q4"), _Value_3, _Value_8, Value.var("q9")),
        MultiSet(Value.var("q7"), _Value_8, Value.var("q9"), _Value_4),
        MultiSet(_Value_8, Value.var("q9"), _Value_13, _Value_6),
        MultiSet(_Value_1, _Value_9, Value.var("q1"), _Value_5),
        MultiSet(Value.var("q4"), _Value_3, _Value_9, _Value_5),
        MultiSet(Value.var("q7"), _Value_4, _Value_9, _Value_5),
        MultiSet(_Value_13, _Value_9, _Value_5, _Value_6),
        MultiSet(Value.var("q3"), _Value_1, _Value_11, _Value_2, Value.var("q1")),
       

In [16]:
function_example.polynomial_multisets_factored.extracted

_Value_1 = polynomial(
    MultiSet(
        MultiSet(Value.var("bp2"), Value.var("q6")),
        MultiSet(Value.var("q3"), Value.var("bp1")),
        MultiSet(Value.var("bp4"), Value.var("q12")),
        MultiSet(Value.var("bp3"), Value.var("q9")),
    )
)
_Value_2 = polynomial(
    MultiSet(
        MultiSet(Value.var("bpp1"), Value.var("q2")),
        MultiSet(Value.var("bpp2"), Value.var("q5")),
        MultiSet(Value.var("bpp3"), Value.var("q8")),
        MultiSet(Value.var("bpp4"), Value.var("q11")),
    )
)
_Value_3 = polynomial(
    MultiSet(
        MultiSet(Value.var("bp2"), Value.var("q5")),
        MultiSet(Value.var("bp3"), Value.var("q8")),
        MultiSet(Value.var("q2"), Value.var("bp1")),
        MultiSet(Value.var("bp4"), Value.var("q11")),
    )
)
_Value_4 = polynomial(
    MultiSet(
        MultiSet(Value.var("q3"), Value.var("bpp1")),
        MultiSet(Value.var("q6"), Value.var("bpp2")),
        MultiSet(Value.var("q9"), Value.var("bpp3")),
        MultiSet(Value.var("q12"), Value.var("bpp4")),
    )
)
_Value_5 = polynomial(
    MultiSet(
        MultiSet(
            Value.from_int(Int(-1)), polynomial(MultiSet(MultiSet(_Value_1, _Value_2)))
        ),
        MultiSet(_Value_3, _Value_4),
    )
)
_Value_6 = polynomial(
    MultiSet(
        MultiSet(Value.var("q4"), Value.var("bp2")),
        MultiSet(Value.var("q7"), Value.var("bp3")),
        MultiSet(Value.var("q10"), Value.var("bp4")),
        MultiSet(Value.var("bp1"), Value.var("q1")),
    )
)
_Value_7 = polynomial(
    MultiSet(
        MultiSet(Value.var("q4"), Value.var("bpp2")),
        MultiSet(Value.var("bpp1"), Value.var("q1")),
        MultiSet(Value.var("q7"), Value.var("bpp3")),
        MultiSet(Value.var("q10"), Value.var("bpp4")),
    )
)
_Value_8 = polynomial(
    MultiSet(
        MultiSet(
            Value.from_int(Int(-1)), polynomial(MultiSet(MultiSet(_Value_6, _Value_4)))
        ),
        MultiSet(_Value_1, _Value_7),
    )
)
_Value_9 = polynomial(
    MultiSet(
        MultiSet(
            Value.from_int(Int(-1)), polynomial(MultiSet(MultiSet(_Value_3, _Value_7)))
        ),
        MultiSet(_Value_6, _Value_2),
    )
)
_Value_10 = polynomial(
    MultiSet(
        MultiSet(_Value_6, _Value_6),
        MultiSet(_Value_3, _Value_3),
        MultiSet(_Value_1, _Value_1),
    )
)
NDArray(
    RecursiveValue(
        polynomial(
            MultiSet(
                MultiSet(_Value_5, _Value_5),
                MultiSet(_Value_8, _Value_8),
                MultiSet(_Value_9, _Value_9),
            )
        )
        / polynomial(MultiSet(MultiSet(_Value_10, _Value_10, _Value_10)))
    )
)

In [17]:
function_example.polynomial.extracted

_Value_1 = (
    Value.var("bp2") * Value.var("q6")
    + Value.var("q3") * Value.var("bp1")
    + Value.var("bp3") * Value.var("q9")
    + Value.var("bp4") * Value.var("q12")
)
_Value_2 = (
    Value.var("bpp1") * Value.var("q2")
    + Value.var("bpp2") * Value.var("q5")
    + Value.var("bpp3") * Value.var("q8")
    + Value.var("bpp4") * Value.var("q11")
)
_Value_3 = (
    Value.var("bp2") * Value.var("q5")
    + Value.var("bp3") * Value.var("q8")
    + Value.var("q2") * Value.var("bp1")
    + Value.var("bp4") * Value.var("q11")
)
_Value_4 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("bp1") * Value.var("q1")
    + Value.var("q10") * Value.var("bp4")
)
_Value_6 = (
    Value.var("q4") * Value.var("bpp2")
    + Value.var("bpp1") * Value.var("q1")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
NDArray(
    RecursiveValue(
        (
            (Value.from_int(Int(-1)) * (_Value_1 * _Value_2) + _Value_3 * _Value_4)
            ** Value.from_int(Int(2))
            + (Value.from_int(Int(-1)) * (_Value_5 * _Value_4) + _Value_1 * _Value_6)
            ** Value.from_int(Int(2))
            + (Value.from_int(Int(-1)) * (_Value_3 * _Value_6) + _Value_5 * _Value_2)
            ** Value.from_int(Int(2))
        )
        / (
            _Value_5 ** Value.from_int(Int(2))
            + _Value_3 ** Value.from_int(Int(2))
            + _Value_1 ** Value.from_int(Int(2))
        )
        ** Value.from_int(Int(3))
    )
)

In [9]:
# frozen = egraph.freeze()
# root = frozen.global_bindings['$GradientBending_distributed']

In [10]:
# egglog.get_callable_args(egglog.get_callable_args(frozen.outputs[egglog.get_callable_args(frozen.outputs[root][0].output)[0]][0].output)[0])

In [11]:
# @egglog.ruleset
# def non_terminate(x: enp.Value, y: enp.Value, z: enp.Value):
#     yield egglog.rewrite((x * y) * z).to(x * (y * z))
#     print(egglog.rewrite(0 * x).to(x))
#     yield egglog.rewrite(0 * x).to(x)

# init = 0 * egglog.constant("a", enp.Value)
# print(init)
# egraph = egglog.EGraph(save_egglog_string=True)
# egraph.register(init)
# egraph.run(non_terminate * 5)
# print(egraph.as_egglog_string)
# egraph.saturate(non_terminate, max=5)a

In [12]:
# x1, x2, x3, x4, x5, x6, x7, x8, x9, x10 = (egglog.constant(f"x{i}", enp.Value) for i in range(1, 11))

# (EXPR := (

# (x1 + x2 / 2 - x3/3 + x4/4)**6 *
#        (x5 + 2*x6 - 3*x7 + 4*x8)**5 * (x9 + x10)**2
#      - (x1 + x2/4 - x3/3 + x4/2)**7 *
#        (x5 + 4*x6 - 3*x7 + 2*x8)**4 * (x9 - x10)**2
# ))

In [13]:
# @egglog.ruleset
# def remove_div_exp(x: enp.Value, y: enp.Value, i: egglog.i64):
#     # Replace div with multiplication
#     yield egglog.rewrite(x / i, subsume=True).to(x * egglog.BigRat(1, i))
#     # Replace power with multiplcation
#     yield egglog.rewrite(x ** i, subsume=True).to(x ** (i -1) * x, i != 1)
#     yield egglog.rewrite(x ** 1, subsume=True).to(x)

# egraph = egglog.EGraph()
# egraph.register(EXPR)
# egraph.run(remove_div_exp.saturate())
# egraph.extract(EXPR)

In [14]:
# egraph.run(distribute.saturate())
# egraph.extract(EXPR)